In [1]:
# from block_group_load_curves import *
# from location_str_to_geoid_mapping import *
# from public_chargers import *
from sklearn.metrics.pairwise import haversine_distances
import folium
from folium.features import GeoJsonTooltip
import pandas as pd
from pathlib import Path
import geopandas as gpd
import numpy as np
import json

In [2]:

# =========================================================
# Find origin and destination energy demand and distances
# =========================================================

DATA_DIR = Path.cwd().parent / "data"

df = pd.read_parquet(DATA_DIR / 'network_analysis/df.parquet')
gdf_subset = pd.read_parquet(DATA_DIR / 'network_analysis/gdf_subset.parquet')
charger_df = pd.read_parquet(DATA_DIR / 'network_analysis/charger_df.parquet')
charger_gdf = gpd.read_parquet(DATA_DIR / 'network_analysis/charger_gdf.parquet')
gdf = gpd.read_parquet(DATA_DIR / 'network_analysis/gdf.parquet')
gdf_for_json = gpd.read_file(DATA_DIR / 'network_analysis/gdf.geojson')

with open((DATA_DIR / 'Geography_Files/location_str_to_geoid_mapping.json'), 'r') as f:
    mapping = json.load(f)

# Flatten column MultiIndex
df.columns = [
    '_'.join(col).strip() if isinstance(col, tuple) else col
    for col in df.columns.values
]

In [3]:

# Merge lat and lon data from GeoPandas df with block group data and map to census track
demand_df = df.merge(gdf_subset, on='geoid_str_', how='left')
# demand_df

In [4]:
gdf_subset.head()

,geoid_str_,INTPTLAT,INTPTLON,NEIGHBOURS
0,"2 (Tract 4232, Alameda, CA)",37.862497,-122.293241,"[1 (Tract 4233, Alameda, CA), 4 (Tract 4231, A..."
1,"1 (Tract 423.36, Orange, CA)",33.545621,-117.699173,"[2 (Tract 423.29, Orange, CA), 2 (Tract 423.19..."
2,"2 (Tract 4043, Alameda, CA)",37.844708,-122.241138,"[2 (Tract 4003, Alameda, CA), 3 (Tract 4043, A..."
3,"1 (Tract 3851, Contra Costa, CA)",37.924964,-122.298942,"[3 (Tract 3901, Contra Costa, CA), 1 (Tract 39..."
4,"2 (Tract 2512, Solano, CA)",38.104862,-122.237457,"[1 (Tract 2510, Solano, CA), 3 (Tract 2512, So..."


In [5]:
print(gdf_subset.columns.tolist())

['geoid_str_', 'INTPTLAT', 'INTPTLON', 'NEIGHBOURS']


In [6]:


# Filter df for data at only 6 pm
demand_df_filtered = demand_df[demand_df['hour_'] == pd.to_timedelta('18:00:00')]

# Drop locations with invalid coordinates
demand_df_filtered = demand_df_filtered.dropna(subset=['INTPTLAT', 'INTPTLON'])

# Take only locations from Alameda County
demand_df_filtered = demand_df_filtered[demand_df_filtered['geoid_str_'].str.contains('Alameda', case=False, na=False)]

# Compute mass
demand_df_filtered['mass'] = demand_df_filtered.loc[:, 'load_curve_MFH_LD_L2':'load_curve_Work_LD_L2'].sum(axis=1)

# Convert to radians for haversine calculation
coords = np.radians(demand_df_filtered[['INTPTLAT', 'INTPTLON']])
dist_matrix = haversine_distances(coords) * 6371

# Change dist matrix from array to pandas df
df_dist = pd.DataFrame(
    dist_matrix,
    columns=demand_df_filtered['geoid_str_'].values
).assign(geoid_str_=demand_df_filtered['geoid_str_'].values)[
    ['geoid_str_'] + demand_df_filtered['geoid_str_'].values.tolist()
]
# Set geo ids as index
df_dist = df_dist.set_index('geoid_str_')

# Map index values to geo id values
index = pd.Series(range(len(df_dist)), index=df_dist.index)


In [7]:
# =========================================================
# Nearest neighbours
# =========================================================

# Take nearest neighbours data from location_str_to_geoid_mapping.py
network_df = gdf_subset.explode('NEIGHBOURS')

# Spatial join of charger location points and census track polygons to categorize each charger
# with its respective census track
charger_loc_gdf = gpd.sjoin(
    charger_gdf,
    gdf[["GEOID_STR", "geometry"]],
    how="left",
    predicate="within"
)
charger_loc_gdf = charger_loc_gdf.drop(columns=["index_right"])
charger_loc_gdf.rename(columns={"GEOID_STR": "geoid_str_"}, inplace=True)

# group chargers by geoid_str_ and aggregate into a list
chargers_per_tract = (
    charger_loc_gdf.groupby("geoid_str_")["ID"]
    .apply(list)
    .reset_index()  # convert back to DataFrame for merging
)

# First merge: origin chargers
network_df = network_df.merge(
    chargers_per_tract,
    how="left",
    left_on="geoid_str_",
    right_on="geoid_str_"
).rename(columns={"ID": "origin_chargers"}) 

# Second merge: destination chargers
network_df = network_df.merge(
    chargers_per_tract,
    how="left",
    left_on="NEIGHBOURS",
    right_on="geoid_str_"
).rename(columns={"ID": "destination_chargers"})

# fix column names
network_df = network_df.drop(columns=["geoid_str__y"])
network_df = network_df.rename(columns={"geoid_str__x": "geoid_str_"})

/var/folders/zv/7h23jkj12tl1zd_4bsx6h9wr0000gn/T/ipykernel_64641/2532565986.py:10: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: None
Right CRS: EPSG:4326

  charger_loc_gdf = gpd.sjoin(


In [8]:
# =========================================================
# Simplify to Alameda County
# =========================================================

# Take only locations from Alameda County
network_df = network_df[network_df['geoid_str_'].str.contains('Alameda', case=False, na=False)]
network_df = network_df[network_df['NEIGHBOURS'].str.contains('Alameda', case=False, na=False)]

# Get demand for each census block
network_df = network_df.merge(demand_df_filtered[['geoid_str_', 'mass']], how='left', on='geoid_str_')
network_df = network_df.rename(columns={'mass':'origin_demand_(kW)'})

# Add destination mass here with df.merge
network_df = pd.merge(network_df, demand_df_filtered[['geoid_str_', 'mass']], left_on='NEIGHBOURS', right_on='geoid_str_', how='left')
network_df = network_df.rename(columns={'mass':'destination_demand_(kW)'})
network_df = network_df.rename(columns={'geoid_str__x': 'geoid_str_'}).drop(columns=['geoid_str__y'])

# Begin Add Haversine distances
# Lookup matrix positions
row_idx = index[network_df['geoid_str_']].values
col_idx = index[network_df['NEIGHBOURS']].values

# Extract distances
network_df['distance_km'] = dist_matrix[row_idx, col_idx]

# Adding location str mapping ids 
network_df['geoid'] = network_df['geoid_str_'].map(mapping)
network_df['neighbor_geoid'] = network_df['NEIGHBOURS'].map(mapping)

network_df.to_parquet(DATA_DIR / 'network_analysis/network_df.parquet')

# =============================================
# End origin-destination analysis
# =============================================

In [9]:
network_df.head()

,geoid_str_,INTPTLAT,INTPTLON,NEIGHBOURS,origin_chargers,destination_chargers,origin_demand_(kW),destination_demand_(kW),distance_km,geoid,neighbor_geoid
0,"2 (Tract 4232, Alameda, CA)",37.862497,-122.293241,"1 (Tract 4233, Alameda, CA)",[333026],NaN,568.288007,274.272001,0.760447,060014232002,060014233001
1,"2 (Tract 4232, Alameda, CA)",37.862497,-122.293241,"4 (Tract 4231, Alameda, CA)",[333026],NaN,568.288007,433.504001,0.668007,060014232002,060014231004
2,"2 (Tract 4232, Alameda, CA)",37.862497,-122.293241,"1 (Tract 4220, Alameda, CA)",[333026],"[161015, 205385, 220218, 220219, 308753, 32132...",568.288007,2937.020025,0.823073,060014232002,060014220001
3,"2 (Tract 4232, Alameda, CA)",37.862497,-122.293241,"3 (Tract 4231, Alameda, CA)",[333026],NaN,568.288007,178.240000,0.667594,060014232002,060014231003
4,"2 (Tract 4232, Alameda, CA)",37.862497,-122.293241,"2 (Tract 4231, Alameda, CA)",[333026],NaN,568.288007,637.020002,0.707463,060014232002,060014231002


### New Section - Calculate travel time and append to network_df

In [10]:
import osmnx as ox
import networkx as nx

print("Loading Alameda County road network...")
G = ox.graph_from_place("Alameda County, California, USA", network_type="drive")
G = ox.speed.add_edge_speeds(G)
G = ox.speed.add_edge_travel_times(G)
print("Network ready.")

Loading Alameda County road network...
Network ready.


In [11]:
#create travel time df of origin destination pairs and respective lat/lon 

tt_df = network_df[['geoid_str_', 'INTPTLAT', 'INTPTLON', 'NEIGHBOURS']].copy()
tt_df.columns = ["orig_geoid_str", "orig_lat", "orig_lon", "dest_geoid_str"]

tt_df = tt_df.merge(
    gdf_subset,
    left_on='dest_geoid_str',  # column in table1
    right_on='geoid_str_',       # column in table2
    how='left'
)

tt_df = tt_df.drop(columns = ["NEIGHBOURS", "geoid_str_"])
tt_df = tt_df.rename(columns={'INTPTLAT': 'dest_lat'})
tt_df = tt_df.rename(columns={'INTPTLON': 'dest_lon'})

In [12]:
# snap OD pair to node on G
tt_df["origin_node"] = ox.distance.nearest_nodes(
    G,
    tt_df["orig_lon"].values,
    tt_df["orig_lat"].values
)

tt_df["dest_node"] = ox.distance.nearest_nodes(
    G,
    tt_df["dest_lon"].values,
    tt_df["dest_lat"].values
)


In [13]:
def compute_tt(row):
    try:
        return nx.shortest_path_length(
            G,
            source=row["origin_node"],
            target=row["dest_node"],
            weight="travel_time"
        )
    except:
        return np.nan  # unreachable

tt_df["travel_time_sec"] = tt_df.apply(compute_tt, axis=1)
tt_df["travel_time_min"] = tt_df["travel_time_sec"] / 60

In [14]:
network_df = network_df.merge(
    tt_df[['orig_geoid_str', 'dest_geoid_str', "travel_time_min", "travel_time_sec"]],
    left_on=['geoid_str_', 'NEIGHBOURS'],  
    right_on=['orig_geoid_str', 'dest_geoid_str'], 
    how='left'                     
)


In [16]:
network_df.head()

,geoid_str_,INTPTLAT,INTPTLON,NEIGHBOURS,origin_chargers,destination_chargers,origin_demand_(kW),destination_demand_(kW),distance_km,geoid,neighbor_geoid,orig_geoid_str,dest_geoid_str,travel_time_min,travel_time_sec
0,"2 (Tract 4232, Alameda, CA)",37.862497,-122.293241,"1 (Tract 4233, Alameda, CA)",[333026],NaN,568.288007,274.272001,0.760447,060014232002,060014233001,"2 (Tract 4232, Alameda, CA)","1 (Tract 4233, Alameda, CA)",1.370000,82.2
1,"2 (Tract 4232, Alameda, CA)",37.862497,-122.293241,"4 (Tract 4231, Alameda, CA)",[333026],NaN,568.288007,433.504001,0.668007,060014232002,060014231004,"2 (Tract 4232, Alameda, CA)","4 (Tract 4231, Alameda, CA)",1.253333,75.2
2,"2 (Tract 4232, Alameda, CA)",37.862497,-122.293241,"1 (Tract 4220, Alameda, CA)",[333026],"[161015, 205385, 220218, 220219, 308753, 32132...",568.288007,2937.020025,0.823073,060014232002,060014220001,"2 (Tract 4232, Alameda, CA)","1 (Tract 4220, Alameda, CA)",1.003333,60.2
3,"2 (Tract 4232, Alameda, CA)",37.862497,-122.293241,"3 (Tract 4231, Alameda, CA)",[333026],NaN,568.288007,178.240000,0.667594,060014232002,060014231003,"2 (Tract 4232, Alameda, CA)","3 (Tract 4231, Alameda, CA)",1.236667,74.2
4,"2 (Tract 4232, Alameda, CA)",37.862497,-122.293241,"2 (Tract 4231, Alameda, CA)",[333026],NaN,568.288007,637.020002,0.707463,060014232002,060014231002,"2 (Tract 4232, Alameda, CA)","2 (Tract 4231, Alameda, CA)",1.261667,75.7


In [17]:
#sanity check
import geopy.distance

for i, row in tt_df.iterrows():
    dist_km = geopy.distance.distance(
        (row['orig_lat'], row['orig_lon']),
        (row['dest_lat'], row['dest_lon'])
    ).km
    if dist_km > 4: 
        print(i, row['orig_geoid_str'], row['dest_geoid_str'], dist_km)

78 1 (Tract 4100, Alameda, CA) 1 (Tract 4301.02, Alameda, CA) 6.883713358397987
126 3 (Tract 4351.03, Alameda, CA) 1 (Tract 4351.03, Alameda, CA) 7.563637788115718
222 1 (Tract 4511.02, Alameda, CA) 2 (Tract 4511.03, Alameda, CA) 4.648600883013519
291 5 (Tract 4507.01, Alameda, CA) 2 (Tract 4411, Alameda, CA) 14.362873124402572
292 5 (Tract 4507.01, Alameda, CA) 1 (Tract 4506.01, Alameda, CA) 12.998801019854733
293 5 (Tract 4507.01, Alameda, CA) 3 (Tract 4507.01, Alameda, CA) 11.477001264218192
294 5 (Tract 4507.01, Alameda, CA) 3 (Tract 4506.01, Alameda, CA) 13.442027942610233
295 5 (Tract 4507.01, Alameda, CA) 1 (Tract 4507.01, Alameda, CA) 11.330634760446715
296 5 (Tract 4507.01, Alameda, CA) 1 (Tract 4351.03, Alameda, CA) 17.963821267482302
297 5 (Tract 4507.01, Alameda, CA) 3 (Tract 4506.07, Alameda, CA) 12.905647514319236
298 5 (Tract 4507.01, Alameda, CA) 2 (Tract 4507.01, Alameda, CA) 12.70404554796642
299 5 (Tract 4507.01, Alameda, CA) 2 (Tract 4507.42, Alameda, CA) 12.6956506